# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze a Croissant-standard FAIR dataset about human extracellular vesicle proteomics, using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Explore available record sets and their fields, referencing all entities strictly by their `@id` as per the Croissant schema. This will help us determine what data is available and how to access it for analysis.


In [ ]:
# List all RecordSets and their @id and field @ids
print("Available RecordSets and fields (referenced by @id):\n")
record_sets = []
# Each record set is referenced by its @id
for rset in metadata.record_sets:
    print(f"RecordSet @id: {rset['@id']}")
    record_sets.append(rset['@id'])
    if hasattr(rset, 'fields'):
        for field in rset.fields:
            print(f"  Field @id: {field['@id']}, name: {field.get('name', 'N/A')}")
    else:
        print("  No fields found.")
    print('-' * 40)

if not record_sets:
    print("No RecordSets found in metadata. Attempting to enumerate records based on mlcroissant auto-discovery.\n")

# Show a few sample records from each RecordSet
for rset_id in record_sets:
    print(f"\nThe first 2 records in RecordSet `{rset_id}`:")
    for i, record in enumerate(dataset.records(record_set=rset_id)):
        print(json.dumps(record, indent=2))
        if i >= 1:
            break


## 3. Data Extraction
Load data from each record set, referencing their `@id`. The columns in the DataFrame will correspond to the fields by their `@id` values, ensuring future-proof, schema-compliant data handling. The example below shows how to extract all data from all record sets.


In [ ]:
# Extract all records from all available record sets into DataFrames
dataframes = dict()
for rset_id in record_sets:
    # Each record is a dict keyed by field @id
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df
    print(f"RecordSet `{rset_id}` - DataFrame columns:")
    print(df.columns.tolist())
    print(df.head(), "\n")

# Pick a record set with tabular data for downstream analysis
if len(dataframes) > 0:
    main_record_set_id = record_sets[0]  # Select the first one for demonstration
else:
    main_record_set_id = None

# If no explicit record sets, try auto-discovering with dataset.records()/dataset.to_dataframe()
if main_record_set_id is None and hasattr(dataset, 'to_dataframe'):
    df = dataset.to_dataframe()
    print("Dataset (auto DataFrame) columns:", df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Next, we'll perform common data processing steps like filtering, normalization, and grouping, all by referencing the relevant field columns by their `@id`. We will choose a numeric field and a grouping field for demonstration based on what is available in the dataset.

In [ ]:
# Attempt EDA on the main tabular RecordSet (identified by @id)
import numpy as np

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    print(f"\nUsing RecordSet @id: {main_record_set_id}")
    print(f"Available columns (all by @id): {df.columns.tolist()}")

    # Try to guess a numeric field (@id) for demonstration
    candidate_numerics = [col for col in df.columns if df[col].dtype in [np.float64, np.int64] or np.issubdtype(df[col].dtype, np.number)]
    if not candidate_numerics and len(df.columns) > 0:
        # Try to infer numerics from column names containing known field names
        candidate_numerics = [col for col in df.columns if 'abundance' in col or 'count' in col or 'MW' in col or 'coverage' in col]
    
    print(f"Candidate numeric fields: {candidate_numerics}")
    if candidate_numerics:
        numeric_field = candidate_numerics[0]
        # Try to coerce field to numeric if needed
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"\nFiltered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a likely categorical field (e.g., description, or any field not numeric)
        candidate_groups = [col for col in df.columns if col not in candidate_numerics]
        if candidate_groups:
            group_field = candidate_groups[0]
            print(f"\nGrouping by field @id: {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field].agg(['mean','std','count'])
            print(grouped.head())
    else:
        print("No obvious numeric field found for EDA. Please inspect the DataFrame columns above.")
else:
    print("No record sets/dataframes found for EDA.")

## 5. Visualization
Visualize numeric field distributions or relationships between fields. Plotting is keyed on fields always referenced by their `@id` according to Croissant best practices.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and candidate_numerics:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} (field @id)")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group_field if set
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(12,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:

- Load structured dataset metadata and records using the Croissant schema and `mlcroissant`.
- List and review record set and field `@id`s.
- Extract and analyze tabular data fully by its schema identifiers, ensuring robust and future-proof data handling.
- Apply exploratory data processing and visualization by referencing all objects using their authoritative `@id`.

This workflow supports easy extension for downstream ML/AI, reproducible science, and compliance with FAIR data standards. You may proceed to select fields and records relevant for your analysis --- always using their `@id` from the Croissant schema for precise, long-lived code!